# DINOv2 新分支：冻结编码器 + 12 类注意力头

这是新的训练实验，不是提交 Notebook，也不承诺超过 0.958。使用全部可用弱标签病例，不再限制为 512 例。
第一阶段冻结 DINOv2 Small，提取每例 3 个方向 × 12 张切片的特征，训练带方向/切片位置参数的逐类别注意力头。暂不微调编码器，先判断此初始化是否有学习价值。

## 添加 3 个 Input

1. [比赛数据](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)
2. [完整 4407 例预处理缓存的成功版本](https://www.kaggle.com/code/openll/rsna20261010/output)
3. [报告弱标签](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels)

若跨账号使用第 2 项，先共享原 Notebook，再添加成功版本的 Output。路径自动在有限目录层级内定位，不递归扫描比赛 MRI。

## 运行

GPU T4 ×2，**Internet On**（下载预训练权重），新建 Notebook 导入本文件，**Save Version → Save & Run All**。
特征提取使用两张 GPU，分类头训练使用 GPU 0。每 40 例打印进度和估计剩余时间；每个 epoch 打印指标。首次全量提取可能较久，ETA 以实际吞吐为准。
MRI 缓存直接读取，不重新解码 570 GB 原数据。切片为灰度重复 RGB，224 像素输入；不恢复原图细节，因此它只是成本较低的第一阶段。

## 验证含义

弱标签按约 70%/15%/15% 分为 train/dev/holdout，实际比例随分组而变化。相同规范化报告、完全相同影像的连通分组不跨集合；与 58 例官方参考病例重合的组不进入弱标签训练。没有患者 ID，不能保证患者级隔离。
最多 20 轮，仅按 dev 宏 AUC 选模型，5 轮未改善则停止。holdout 在选定后只评估一次，之后若据它调参，它也会变成开发集。58 例只报告参考成绩，不用于选轮次。
这些是弱标签指标，不等价于排行榜；没有 R6/R7 同一批病例的无泄漏预测，本次不能证明融合收益。当前不会生成 submission.csv。

## 跑完交付

下载 Output 中 `dino-branch-v1` 文件夹的 `summary.json`、`metrics.jsonl`、`label_coverage.json`、`config.json` 和三个 predictions.csv（或整个 Output ZIP）。保留 best_head.pt、dino_encoder.pt、特征和 split.csv，方便后续分析和复用。
不要覆盖当前 0.958 成功提交。

初始化来源：[timm DINOv2 Small 模型卡](https://huggingface.co/timm/vit_small_patch14_dinov2.lvd142m)，[DINOv2 官方代码](https://github.com/facebookresearch/dinov2)。


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','timm==1.0.29'],check=True)
import torch
assert torch.cuda.is_available(), '请启用 GPU'
print('GPU:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)


## 定位输入
只检查预期文件；如果有多个候选则停止并显示路径，避免误用其他缓存。

In [ ]:
from pathlib import Path
import json
ROOT=Path('/kaggle/input')
DATA=ROOT/'competitions/rsna-knee-abnormality-detection'
if not (DATA/'train.csv').is_file(): DATA=ROOT/'rsna-knee-abnormality-detection'
assert (DATA/'train.csv').is_file(), '请添加比赛数据'
dirs=[ROOT]
front=[ROOT]
for _ in range(4):
    nxt=[]
    for parent in front:
        for p in parent.iterdir():
            if p.is_dir() and p.name not in ['competitions','rsna-knee-abnormality-detection','train_series','test_series']:
                nxt.append(p)
    dirs.extend(nxt);front=nxt
caches=[p for p in dirs if p.name=='rsna-cache-smoke' and (p/'config.json').is_file()]
labels=[p/'report_labels_v2.csv' for p in dirs if (p/'report_labels_v2.csv').is_file()]
assert len(caches)==1, f'缓存候选：{caches}；请只挂载完整预处理版本'
assert len(labels)==1, f'弱标签候选：{labels}'
CACHE,WEAK=caches[0],labels[0]
cfg=json.loads((CACHE/'config.json').read_text())
assert cfg['size']==224 and cfg['slices']==12 and cfg['split']=='train',cfg
OUT=Path('/kaggle/working/dino-branch-v1')
print('DATA',DATA,'CACHE',CACHE,'WEAK',WEAK,'OUTPUT',OUT,sep='\n',flush=True)


## 写入训练脚本

In [ ]:
SOURCE='"""Frozen DINOv2 MRI branch: grouped development and one-time weak holdout."""\nimport argparse\nimport hashlib\nimport json\nimport random\nimport time\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\n\nUID = \'StudyInstanceUID\'\nLABELS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nARCH = \'vit_small_patch14_dinov2.lvd142m\'\n\n\ndef digest(p):\n    return hashlib.sha256(Path(p).read_bytes()).hexdigest()\n\n\ndef report_hash(s):\n    s = \' \'.join(unicodedata.normalize(\'NFKC\', str(s)).casefold().split())\n    return hashlib.sha256(s.encode()).hexdigest() if s else \'\'\n\n\ndef make_split(official, weak, image_hashes, seed=20261019):\n    from sklearn.model_selection import GroupShuffleSplit\n    for df in [official, weak]:\n        assert df[UID].notna().all() and df[UID].is_unique, \'Invalid study IDs\'\n    table = official[[UID,\'Report\']+LABELS].copy().reset_index(drop=True)\n    gold = table[LABELS].notna().all(1)\n    empty = table[LABELS].isna().all(1)\n    assert (gold | empty).all(), \'Unexpected partially official-labelled rows\'\n    parent = list(range(len(table)))\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]; i = parent[i]\n        return i\n    seen = {}\n    reports = table.Report.fillna(\'\').map(report_hash)\n    for i, uid in enumerate(table[UID]):\n        for key in [(\'report\',reports[i]),(\'image\',image_hashes[uid])]:\n            if not key[1]: continue\n            if key in seen: parent[find(i)] = find(seen[key])\n            else: seen[key] = i\n    table[\'group\'] = [find(i) for i in range(len(table))]\n    gold_groups = set(table.loc[gold,\'group\'])\n    table[\'split\'] = \'excluded\'\n    table.loc[gold,\'split\'] = \'gold_reference\'\n    eligible = empty & ~table.group.isin(gold_groups) & reports.ne(\'\')\n    aligned = weak.set_index(UID).reindex(table[UID])\n    for c in LABELS:\n        v = aligned[c+\'__verdict\'].astype(\'string\').str.strip().str.upper()\n        assert (v.isna() | v.isin([\'YES\',\'NO\',\'UNK\'])).all(), c\n        vals = v.map({\'YES\':1.,\'NO\':0.,\'UNK\':np.nan}).to_numpy(dtype=float,na_value=np.nan)\n        table.loc[empty,c] = vals[empty.to_numpy()]\n    eligible &= table[LABELS].notna().any(axis=1)\n    ids = np.flatnonzero(eligible)\n    a,b = next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=seed).split(ids,groups=table.loc[ids,\'group\']))\n    pool,hold = ids[a],ids[b]\n    a,b = next(GroupShuffleSplit(n_splits=1,test_size=.15/.85,random_state=seed+1).split(pool,groups=table.loc[pool,\'group\']))\n    for name, rows in [(\'train\',pool[a]),(\'dev\',pool[b]),(\'holdout\',hold)]: table.loc[rows,\'split\'] = name\n    sets = [set(table.loc[table.split.eq(s),\'group\']) for s in [\'train\',\'dev\',\'holdout\',\'gold_reference\']]\n    assert all(not sets[i]&sets[j] for i in range(4) for j in range(i))\n    return table.drop(columns=\'Report\')\n\n\nclass Images(Dataset):\n    def __init__(self, ids, cache): self.ids,self.cache = list(ids),Path(cache)\n    def __len__(self): return len(self.ids)\n    def __getitem__(self,i):\n        uid = self.ids[i]\n        with np.load(self.cache/(uid+\'.npz\'),allow_pickle=False) as z:\n            x,m = z[\'x\'].copy(),z[\'mask\'].astype(bool)\n        assert x.shape==(3,12,224,224) and x.dtype==np.uint8, (uid,x.shape,x.dtype)\n        assert m.shape==(3,) and m.any(), uid\n        h=hashlib.sha256(x.tobytes()+m.tobytes()).hexdigest()\n        return torch.from_numpy(x),torch.from_numpy(m),uid,h\n\n\ndef extract(ids,cache,out,device):\n    import timm\n    fdir=out/\'features\'; fdir.mkdir(exist_ok=True)\n    model=timm.create_model(ARCH,pretrained=True,num_classes=0,img_size=224).eval().to(device)\n    cfg=timm.data.resolve_model_data_config(model)\n    torch.save(model.cpu().state_dict(),out/\'dino_encoder.pt\'); model.to(device)\n    config={\'arch\':ARCH,\'size\':224,\'slices\':12,\'planes\':3,\'mean\':list(cfg[\'mean\']),\'std\':list(cfg[\'std\']),\n            \'encoder_sha256\':digest(out/\'dino_encoder.pt\'),\'cache_config_sha256\':digest(cache/\'config.json\'),\n            \'timm\':timm.__version__,\'torch\':torch.__version__,\'input\':\'uint8 / 255; grayscale repeated RGB; no spatial crop\'}\n    # Never silently reuse features from a different encoder or preprocessing recipe.\n    signature=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()\n    config[\'signature\']=signature\n    (out/\'feature_config.json\').write_text(json.dumps(config,indent=2))\n    hashes={}; todo=[]\n    for uid in ids:\n        p=fdir/(uid+\'.npz\')\n        if p.exists():\n            with np.load(p,allow_pickle=False) as z:\n                if str(z[\'signature\'])==signature:\n                    hashes[uid]=str(z[\'image_hash\']); continue\n        todo.append(uid)\n    mean=torch.tensor(cfg[\'mean\'],device=device)[None,:,None,None]\n    std=torch.tensor(cfg[\'std\'],device=device)[None,:,None,None]\n    if torch.cuda.device_count()>1: model=nn.DataParallel(model)\n    loader=DataLoader(Images(todo,cache),batch_size=4,num_workers=2,pin_memory=True)\n    start=time.time(); done=0\n    with torch.inference_mode():\n        for x,m,uids,hh in loader:\n            b=len(uids)\n            images=x.to(device).reshape(-1,1,224,224).float().div_(255).expand(-1,3,-1,-1)\n            images=(images-mean)/std\n            with torch.autocast(\'cuda\',dtype=torch.float16): f=model(images)\n            f=f.float().reshape(b,3,12,-1).cpu().numpy().astype(\'float16\')\n            assert np.isfinite(f).all()\n            for j,uid in enumerate(uids):\n                p=fdir/(uid+\'.npz\'); tmp=p.with_suffix(\'.tmp.npz\')\n                np.savez_compressed(tmp,f=f[j],mask=m[j].numpy(),image_hash=hh[j],signature=signature)\n                tmp.replace(p);hashes[uid]=hh[j]\n            done+=b\n            if done%40==0 or done==len(todo):\n                elapsed=time.time()-start\n                print(f\'FEATURES {done}/{len(todo)} elapsed={elapsed/60:.1f} min ETA={(len(todo)-done)*elapsed/max(done,1)/60:.1f} min\',flush=True)\n    del model;torch.cuda.empty_cache()\n    return hashes,config\n\n\nclass Features(Dataset):\n    def __init__(self,df,folder):\n        self.df=df.reset_index(drop=True); self.folder=folder\n    def __len__(self): return len(self.df)\n    def __getitem__(self,i):\n        row=self.df.iloc[i]\n        with np.load(self.folder/(row[UID]+\'.npz\'),allow_pickle=False) as z:\n            f=torch.from_numpy(z[\'f\'].astype(\'float32\'));m=torch.from_numpy(z[\'mask\'].copy())\n        y=torch.tensor(row[LABELS].to_numpy(dtype=\'float32\'))\n        return f,m,y\n\n\nclass Head(nn.Module):\n    def __init__(self,dim=384):\n        super().__init__()\n        self.proj=nn.Sequential(nn.LayerNorm(dim),nn.Linear(dim,128),nn.GELU())\n        self.position=nn.Parameter(torch.zeros(1,3,12,128))\n        self.attn=nn.Linear(128,12)\n        self.value=nn.Linear(128,12)\n        self.bias=nn.Parameter(torch.zeros(12))\n        self.drop=nn.Dropout(.2)\n    def forward(self,f,mask):\n        h=self.proj(f)+self.position\n        b=h.shape[0]; h=self.drop(h).reshape(b,36,128)\n        good=mask[:,:,None].expand(-1,-1,12).reshape(b,36,1)\n        a=self.attn(h).masked_fill(~good,-1e4).softmax(1)\n        return (a*self.value(h)).sum(1)+self.bias\n\n\ndef loss_fn(z,y):\n    valid=torch.isfinite(y)\n    loss=nn.functional.binary_cross_entropy_with_logits(z,torch.nan_to_num(y),reduction=\'none\')\n    counts=valid.sum(0); usable=counts>0\n    return ((loss*valid).sum(0)/counts.clamp_min(1))[usable].mean()\n\n\ndef metrics(y,p):\n    from sklearn.metrics import roc_auc_score\n    result={}\n    for j,c in enumerate(LABELS):\n        m=np.isfinite(y[:,j]); truth=y[m,j]\n        result[c]=float(roc_auc_score(truth,p[m,j])) if len(np.unique(truth))==2 else None\n    available=[v for v in result.values() if v is not None]\n    return {\'macro_auc\':float(np.mean(available)) if available else None,\'valid_classes\':len(available),\'per_class\':result}\n\n\n@torch.inference_mode()\ndef evaluate(model,loader,device):\n    model.eval(); yy=[]; pp=[]\n    for f,m,y in loader:\n        pp.append(model(f.to(device),m.to(device)).sigmoid().cpu().numpy());yy.append(y.numpy())\n    y,p=np.concatenate(yy),np.concatenate(pp)\n    return metrics(y,p),p\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    for name in [\'data\',\'weak\',\'cache\',\'out\']: ap.add_argument(\'--\'+name,type=Path,required=True)\n    ap.add_argument(\'--epochs\',type=int,default=20);a=ap.parse_args()\n    assert torch.cuda.is_available(),\'Enable GPU\'\n    seed=20261019;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)\n    torch.set_num_threads(4)\n    a.out.mkdir(parents=True,exist_ok=True)\n    if (a.out/\'summary.json\').exists(): raise RuntimeError(\'Completed experiment exists; use a fresh output directory\')\n    started=time.time()\n    official=pd.read_csv(a.data/\'train.csv\',dtype={UID:str});weak=pd.read_csv(a.weak,dtype={UID:str})\n    assert official[UID].is_unique and official[UID].notna().all()\n    for c in LABELS:\n        assert official[c].dropna().isin([0,1]).all()\n        assert c+\'__verdict\' in weak\n    assert all((a.cache/(u+\'.npz\')).is_file() for u in official[UID]),\'Incomplete cache; mount full 4407-study output\'\n    hashes,fcfg=extract(official[UID],a.cache,a.out,\'cuda\')\n    split=make_split(official,weak,hashes,seed)\n    split.to_csv(a.out/\'split.csv\',index=False)\n    coverage={s:{c:{\'positive\':int((d[c]==1).sum()),\'negative\':int((d[c]==0).sum()),\'unknown\':int(d[c].isna().sum())} for c in LABELS} for s,d in split.groupby(\'split\')}\n    (a.out/\'label_coverage.json\').write_text(json.dumps(coverage,indent=2))\n    counts=split.split.value_counts().to_dict();print(\'SPLITS\',counts,flush=True)\n    for c,v in coverage[\'dev\'].items():\n        if not v[\'positive\'] or not v[\'negative\']: raise RuntimeError(f\'Dev lacks both classes: {c}; do not select a different seed based on results\')\n    config={\'seed\':seed,\'epochs_max\':a.epochs,\'patience\':5,\'lr\':.001,\'weight_decay\':.01,\'backbone_frozen\':True,\n            \'selection\':\'dev weak macro AUC only\',\'feature_config\':fcfg,\'official_sha256\':digest(a.data/\'train.csv\'),\n            \'weak_sha256\':digest(a.weak),\'script_sha256\':digest(__file__),\'split_sha256\':digest(a.out/\'split.csv\'),\n            \'grouping\':\'connected components of normalized exact report and exact image hashes; not patient-level IDs\',\n            \'weak_holdout_is_clinical_ground_truth\':False,\'gold_independent_test\':False,\'baseline_complementarity_tested\':False}\n    (a.out/\'config.json\').write_text(json.dumps(config,indent=2))\n    loaders={s:DataLoader(Features(split[split.split.eq(s)],a.out/\'features\'),batch_size=64,shuffle=s==\'train\',num_workers=2) for s in [\'train\',\'dev\',\'holdout\',\'gold_reference\']}\n    model=Head().cuda();opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)\n    best=-float(\'inf\'); stale=0\n    for epoch in range(1,a.epochs+1):\n        model.train();total=0.;steps=0\n        for f,m,y in loaders[\'train\']:\n            opt.zero_grad(set_to_none=True)\n            loss=loss_fn(model(f.cuda(),m.cuda()),y.cuda())\n            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()\n            total+=loss.item();steps+=1\n        score,_=evaluate(model,loaders[\'dev\'],\'cuda\')\n        record={\'epoch\':epoch,\'loss\':total/steps,\'dev\':score}\n        with (a.out/\'metrics.jsonl\').open(\'a\') as h:h.write(json.dumps(record)+\'\\n\')\n        print(json.dumps(record),flush=True)\n        if score[\'macro_auc\']>best:\n            best=score[\'macro_auc\'];stale=0\n            torch.save({\'state_dict\':model.state_dict(),\'epoch\':epoch,\'labels\':LABELS,\'feature_signature\':fcfg[\'signature\']},a.out/\'best_head.pt\')\n        else:stale+=1\n        if stale>=5:break\n    ckpt=torch.load(a.out/\'best_head.pt\',map_location=\'cuda\',weights_only=True);model.load_state_dict(ckpt[\'state_dict\'])\n    scores={}\n    # The holdout and gold are first evaluated only after the dev-selected model is frozen.\n    for name in [\'dev\',\'holdout\',\'gold_reference\']:\n        scores[name],pred=evaluate(model,loaders[name],\'cuda\')\n        rows=split.loc[split.split.eq(name),[UID,\'group\']+LABELS].reset_index(drop=True)\n        rows=rows.rename(columns={c:\'target::\'+c for c in LABELS})\n        for j,c in enumerate(LABELS):rows[\'pred::\'+c]=pred[:,j]\n        rows.to_csv(a.out/(name+\'_predictions.csv\'),index=False)\n    result={\'status\':\'training_complete\',\'selected_epoch\':ckpt[\'epoch\'],\'counts\':counts,\'metrics\':scores,\n            \'elapsed_seconds\':time.time()-started,\'submission_ready\':False,\'complementarity_with_r6_r7_unknown\':True}\n    (a.out/\'summary.json\').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2),flush=True)\n\n\nif __name__==\'__main__\':main()\n'
SCRIPT=Path('/kaggle/working/dino_branch_train.py')
SCRIPT.write_text(SOURCE,encoding='utf8')


## 提取特征并训练
本单元格需要 GPU。日志持续显示，并保存到 Output。

In [ ]:
import os
env=dict(os.environ,PYTHONUNBUFFERED='1',HF_HOME='/kaggle/working/hf-dino-cache')
cmd=[sys.executable,'-u',str(SCRIPT),'--data',str(DATA),'--weak',str(WEAK),'--cache',str(CACHE),'--out',str(OUT)]
with Path('/kaggle/working/dino-training.log').open('w') as log:
    with subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        rc=proc.wait()
assert rc==0, '训练失败，查看 dino-training.log，不能把本次输出当作成功结果'
print((OUT/'summary.json').read_text(),flush=True)
